# 5-3절 연습 문제 풀이

이 노트북은 5-3절 연습 문제(5-9 ~ 5-12)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch05/05-03_example.ipynb`를 참고한다.
- CIFAR-10, MNIST 데이터셋은 저장소 규약에 따라 `download/` 디렉터리에 저장한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import copy
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = '../../download'
DATA_DIR = '../../data'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'학습 장치: {device}')

학습 장치: cuda


In [2]:
# 본문 [코드 5-12], [코드 5-14], [코드 5-15]와 같은 데이터 준비 과정
CIFAR10_MEAN = (0.5, 0.5, 0.5)
CIFAR10_STD = (0.5, 0.5, 0.5)
BATCH_SIZE = 128

transform_chain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

augmented_set = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=train_transform)
plain_set = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=transform_chain)
test_set = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform_chain)

train_set, _ = random_split(augmented_set, [40000, 10000],
                            generator=torch.Generator().manual_seed(SEED))
_, valid_set = random_split(plain_set, [40000, 10000],
                            generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

class_names = datasets.CIFAR10(root=DATA_ROOT, train=False, download=False).classes
print(f'클래스: {class_names}')

클래스: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
# 본문 [코드 5-18]의 학습 함수를 그대로 사용한다
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, sample_size = 0.0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * inputs.size(0)
        sample_size += inputs.size(0)
    return loss_sum / sample_size

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct_size, sample_size = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss_sum += criterion(outputs, labels).item() * inputs.size(0)
        correct_size += (outputs.argmax(dim=1) == labels).sum().item()
        sample_size += inputs.size(0)
    return loss_sum / sample_size, correct_size / sample_size

def train_with_early_stopping(model, train_loader, valid_loader, criterion, optimizer,
                              epochs, patience, device, scheduler=None, verbose_every=5):
    model.to(device)
    best_valid_loss, best_params, best_epoch, patience_counter = float('inf'), None, 0, 0
    history = []
    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, criterion, device)
        history.append((epoch, train_loss, valid_loss, valid_accuracy))
        if scheduler is not None:
            scheduler.step()
        if epoch % verbose_every == 0 or epoch == 1:
            print(f'  에포크 {epoch:3d} | 훈련 손실 {train_loss:.4f} | '
                  f'검증 손실 {valid_loss:.4f} | 검증 정확도 {valid_accuracy * 100:.2f}%')
        if valid_loss < best_valid_loss:
            best_valid_loss, best_epoch, patience_counter = valid_loss, epoch, 0
            best_params = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  조기 종료: 에포크 {epoch} (최적 에포크 {best_epoch})')
                break
    if best_params is not None:
        model.load_state_dict(best_params)
    return best_epoch, history

In [4]:
# 본문 [코드 5-13], [코드 5-16]의 모델 클래스
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.conv_block(x)

class CIFAR10ConvClassifier(nn.Module):
    def __init__(self, dropout_p=0.5):
        super().__init__()
        self.conv_block_1 = ConvBlock(3, 16, kernel_size=5, stride=1, padding=2)
        self.conv_block_2 = ConvBlock(16, 32, kernel_size=3, stride=1, padding=1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

EPOCHS = 100
PATIENCE = 10
LEARNING_RATE = 0.001

print('본문 모델(드롭아웃 p=0.5) 학습')
torch.manual_seed(SEED)
base_model = CIFAR10ConvClassifier(dropout_p=0.5)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(base_model.parameters(), lr=LEARNING_RATE)
base_epoch, base_history = train_with_early_stopping(
    base_model, train_loader, valid_loader, criterion, optimizer, EPOCHS, PATIENCE, device)
_, base_accuracy = evaluate(base_model, test_loader, criterion, device)
print(f'본문 모델 평가 정확도: {base_accuracy * 100:.2f}%')

본문 모델(드롭아웃 p=0.5) 학습


  에포크   1 | 훈련 손실 1.6011 | 검증 손실 1.3207 | 검증 정확도 53.77%


  에포크   5 | 훈련 손실 1.0310 | 검증 손실 0.9526 | 검증 정확도 66.12%


  에포크  10 | 훈련 손실 0.8435 | 검증 손실 0.8174 | 검증 정확도 70.83%


  에포크  15 | 훈련 손실 0.7407 | 검증 손실 0.7967 | 검증 정확도 72.20%


...

  에포크  25 | 훈련 손실 0.5972 | 검증 손실 0.7770 | 검증 정확도 73.38%


  에포크  30 | 훈련 손실 0.5464 | 검증 손실 0.7980 | 검증 정확도 74.00%


  에포크  35 | 훈련 손실 0.5036 | 검증 손실 0.8694 | 검증 정확도 72.78%


  조기 종료: 에포크 36 (최적 에포크 26)


본문 모델 평가 정확도: 73.34%


## 연습 문제 5-9

> CIFAR-10 분류기 모델을 사용해 깃허브 저장소의 data 디렉터리에 있는 cat.jpg 파일 속 이미지의 클래스를 예측해 보자.
> 이미지 파일은 Pillow 라이브러리의 `PIL.Image.open('cat.jpg')` 함수를 호출해 Pillow 객체로 불러올 수 있다.
> 불러온 Pillow 객체는 `torchvision.transforms`의 변환 기능을 사용해 CIFAR-10 분류기 모델에 입력할 수 있는 형태로
> 변환해 사용해야 한다. 모델의 예측 결과가 예상과 다르다면 그 이유가 무엇인지 추측해 보자.

In [5]:
# 원본 이미지 확인
cat_image = Image.open(f'{DATA_DIR}/cat.jpg')
print(f'원본 이미지 크기: {cat_image.size}, 모드: {cat_image.mode}')

# CIFAR-10 모델에 입력하려면 (3, 32, 32) 형태로 맞춰야 한다
#   Resize: 32x32 로 축소, ToTensor: 텐서 변환, Normalize: 본문과 같은 기준으로 표준화
predict_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
cat_tensor = predict_transform(cat_image).unsqueeze(0)   # (1, 3, 32, 32)
print(f'변환 후 텐서 형태: {tuple(cat_tensor.shape)}')

원본 이미지 크기: (895, 603), 모드: RGB
변환 후 텐서 형태: (1, 3, 32, 32)


In [6]:
def predict_top(model, input_tensor, top_k=3):
    model.eval()
    with torch.no_grad():
        logits = model(input_tensor.to(device))
        probabilities = torch.softmax(logits, dim=1)[0]
    values, indices = probabilities.topk(top_k)
    return [(class_names[i], v.item() * 100) for v, i in zip(values, indices)]

print('cat.jpg 예측 결과 (상위 3개)')
for name, probability in predict_top(base_model, cat_tensor):
    print(f'  {name:>12}: {probability:5.2f}%')

cat.jpg 예측 결과 (상위 3개)
           cat: 79.85%
          frog: 13.48%
          bird:  2.37%


In [7]:
# 비교 1: 가운데를 정사각형으로 잘라 낸 후 축소하면 결과가 달라질까?
crop_transform = transforms.Compose([
    transforms.CenterCrop(min(cat_image.size)),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
print('가운데를 잘라 낸 후 예측')
for name, probability in predict_top(base_model, crop_transform(cat_image).unsqueeze(0)):
    print(f'  {name:>12}: {probability:5.2f}%')

# 비교 2: 같은 모델이 CIFAR-10 의 고양이 이미지는 얼마나 맞히는지 확인
cat_index = class_names.index('cat')
base_model.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predicted = base_model(inputs).argmax(dim=1)
        mask = labels == cat_index
        correct += (predicted[mask] == cat_index).sum().item()
        total += mask.sum().item()
print()
print(f'CIFAR-10 평가 데이터셋의 고양이 분류 정확도: {correct / total * 100:.2f}% ({correct}/{total})')

가운데를 잘라 낸 후 예측
           cat: 30.24%
          ship: 29.22%
      airplane: 24.55%



CIFAR-10 평가 데이터셋의 고양이 분류 정확도: 53.60% (536/1000)


### 풀이 해설

**변환 과정**

CIFAR-10 분류기는 `(B, 3, 32, 32)` 형태의, 본문 기준으로 표준화된 텐서만 입력받는다.
따라서 일반 사진을 넣으려면 세 단계를 거쳐야 한다.

1. `transforms.Resize((32, 32))` — 크기를 32×32로 맞춘다.
2. `transforms.ToTensor()` — Pillow 객체를 `(3, H, W)` 형태의 텐서로 바꾸고 값을 0~1로 만든다.
3. `transforms.Normalize(...)` — **학습할 때와 같은 평균, 표준편차로 표준화한다.**

세 번째가 특히 중요하다. 학습 때와 다른 기준으로 표준화하면 모델이 전혀 다른 분포의 입력을 받게 되어 예측이 무너진다.
그리고 모델은 배치 차원을 기대하므로 `unsqueeze(0)`으로 차원을 하나 더 만들어야 한다.

**예측 결과 — 예상 밖으로 잘 맞힌다**

```
cat.jpg 예측 결과 (상위 3개)
           cat: 79.85%
          frog: 13.48%
          bird:  2.37%
```

**고양이로 정확히 분류했고 확률도 80%에 가깝다.** 문제 지문은 "예측 결과가 예상과 다르다면"이라며
실패를 예고하는데, 실제로는 성공한 셈이다.

그런데 이 성공을 곧이곧대로 받아들이면 안 된다. **같은 모델이 CIFAR-10의 고양이는 53.6%밖에 못 맞힌다.**
열 개 클래스 중 최하위이고, 전체 정확도 73%에 한참 못 미친다.
**평가 데이터셋에서 절반밖에 못 맞히는 클래스를, 처음 보는 사진에서는 80% 확신으로 맞혔다.**
이 어긋남이 이 문제에서 생각해 볼 지점이다.

**왜 이런 일이 생기는가.** `cat.jpg`가 **CIFAR-10의 고양이 이미지와 성격이 비슷했기 때문**일 가능성이 높다.
CIFAR-10의 이미지는 대상이 화면을 거의 채우고 배경이 단순한데, 이 사진도 그런 구도다.
반면 CIFAR-10 평가 데이터셋의 고양이 중에는 웅크려 있거나 일부만 보이는 등 어려운 이미지가 많다.
**한 장의 성공은 통계가 아니다.**

**입력을 조금 바꾸면 바로 흔들린다**

```
가운데를 잘라 낸 후 예측
           cat: 30.24%
          ship: 29.22%
      airplane: 24.55%
```

가로세로 비율을 지키려고 가운데를 정사각형으로 잘라 냈을 뿐인데 **확률이 80%에서 30%로 떨어지고,
`ship`과 `airplane`이 거의 같은 점수로 따라붙는다.** 사실상 찍는 것과 다름없는 상태다.

같은 고양이 사진인데 **어느 부분을 넣느냐에 따라 답이 뒤집힐 만큼 모델이 불안정하다.**
앞의 80%가 실력이 아니라 운에 가까웠다는 것을 이 비교가 보여 준다.

**정리하면** 이 문제가 알려 주는 것은 두 가지다.

1. **훈련 데이터의 분포를 벗어난 입력에 모델이 약하다.** 실무에서 '평가 데이터셋에서는 좋았는데
   실제로는 안 된다'는 문제가 대부분 여기서 온다.
2. **한 장의 결과로 성능을 판단하면 안 된다.** 맞혔든 틀렸든, 입력을 조금 바꿔 보고 확률까지 함께 봐야
   모델이 실제로 무엇을 아는지 알 수 있다.

이를 제대로 다루려면 7장의 전이 학습처럼 훨씬 다양한 이미지로 학습한 모델이 필요하다.

### 문제 검토

- **적절성: 적합. 이 장을 마무리하는 문제로 잘 맞는다.** 지금까지 만든 모델을 **책 밖의 데이터**에 처음 써 보게 한다.
- **★ [검토] 그런데 지문이 전제하는 실패가 실제로는 일어나지 않는다.**
  "모델의 예측 결과가 예상과 다르다면 그 이유가 무엇인지 추측해 보자"는 문장은 **오분류를 전제**하는데,
  실제로 돌려 보면 **고양이로 79.85% 확률로 정확히 맞힌다**(위 실행 결과).
  실패를 기대한 독자는 "잘 되는데?" 하고 아무 생각 없이 넘어가게 된다. 이 문제에서 가장 아쉬운 점이다.

  다만 파고들 거리는 충분히 있다. **같은 모델이 CIFAR-10의 고양이는 53.6%밖에 못 맞히고**,
  **가운데를 잘라 넣으면 확률이 30%로 떨어져 `ship`과 뒤섞인다.**
  즉 '맞혔지만 믿을 수 없다'가 정확한 상태다. → **물음을 '맞혔는가'가 아니라 '얼마나 믿을 만한가'로 바꾸면**
  이 문제가 의도한 교훈이 결과와 관계없이 살아난다. 아래 윤문안이 그 방향이다.
- **[검토] 변환 단계에 대한 힌트가 조금 부족하다.** 지문은 "`torchvision.transforms`의 변환 기능을 사용해"라고만 하는데,
  실제로는 **크기 조정(`Resize`)과 학습 때와 같은 기준의 표준화(`Normalize`)** 두 가지가 모두 필요하다.
  특히 표준화를 빠뜨리면 결과가 크게 어긋나는데, 독자는 그것이 자기 실수 때문인지 모델 한계 때문인지 구분하지 못한다.
  이 문제의 초점은 '모델의 한계'에 있으므로, 변환에서 헤매지 않도록 두 변환을 짚어 주는 편이 낫다.
- **[검토] 원인 추측의 실마리를 주면 좋겠다.** 같은 모델이 CIFAR-10의 고양이는 얼마나 맞히는지 확인해 보게 하면,
  '내 변환이 잘못됐나'와 '모델이 원래 못한다'를 구분할 수 있어 추측이 근거 있는 결론으로 바뀐다.
- **[검토] 파일 경로 표기.** 지문의 `PIL.Image.open('cat.jpg')`는 현재 디렉터리에 파일이 있다고 가정한 형태다.
  저장소에서는 `data/` 디렉터리에 있으므로 실행 위치에 따라 경로를 맞춰야 한다. 앞에서 "깃허브 저장소의 data 디렉터리"라고
  밝혔으니 큰 문제는 아니지만, 예시를 `PIL.Image.open('../../data/cat.jpg')`처럼 쓰면 더 친절하다.

**윤문안**

> **5-9**. CIFAR-10 분류기 모델을 사용해 깃허브 저장소의 data 디렉터리에 있는 cat.jpg 파일 속 이미지의 클래스를 예측해 보자.
> 이미지 파일은 Pillow 라이브러리의 `PIL.Image.open()` 함수를 호출해 Pillow 객체로 불러올 수 있다. 불러온 Pillow 객체는
> `torchvision.transforms`의 크기 조정 변환과 표준화 변환을 사용해 CIFAR-10 분류기 모델에 입력할 수 있는 형태로 변환해
> 사용해야 한다. 이때 표준화의 기준은 학습에 사용한 것과 같아야 한다. 모델의 예측 결과가 예상과 다르다면 그 이유가 무엇인지
> 추측해 보자. 그리고 예측이 얼마나 믿을 만한지 다음 두 가지로 확인해 보자.
> - 같은 모델이 CIFAR-10 평가 데이터셋의 고양이 이미지는 얼마나 잘 분류하는가?
> - 사진의 가운데를 정사각형으로 잘라 낸 후 예측하면 결과가 어떻게 달라지는가?

## 연습 문제 5-10

> [코드 5-17]의 완전 연결 계층에는 드롭아웃이 적용되어 있다.
> - 만약 드롭아웃의 `p` 인자의 값을 0.5에서 0.9로 극단적으로 높인다면, 학습 과정과 모델의 최종 성능에 어떤 영향을 미칠지 추측해 보자.
> - 이 드롭아웃은 분류기의 은닉층과 출력층 사이에 적용되어 있는데, 이는 일반적으로 드롭아웃이 적용되는 위치이다.
>   드롭아웃을 합성곱 계층이 아닌 완전 연결 계층의 은닉층과 출력층 사이에 주로 적용하는 이유를 '특징의 복잡도' 관점에서 설명해 보자.

### 예상

`p=0.9`는 은닉층 뉴런 256개 중 평균 **230개 정도를 매번 꺼 버린다**는 뜻이다. 남는 것은 26개 안팎이다.
여기서 두 가지를 예상할 수 있다.

1. **학습이 느려지고 불안정해진다.** 매 배치마다 살아남는 뉴런이 크게 달라지므로 훈련 손실이 잘 내려가지 않고 출렁인다.
2. **최종 성능이 떨어진다.** 26개 뉴런만으로 10개 클래스를 구분해야 하니 모델의 표현력 자체가 부족해진다.
   과적합을 막으려다 **과소적합**으로 넘어가는 경우다.

다만 한 가지 흥미로운 점이 있다. 드롭아웃은 평가 모드에서는 꺼지므로, **검증 손실은 의외로 나쁘지 않을 수도 있다.**
훈련 손실과 검증 손실의 관계가 평소와 반대로 나타날지도 모른다. 실제로 확인해 보자.

In [8]:
print('드롭아웃 p=0.9 모델 학습')
torch.manual_seed(SEED)
high_dropout_model = CIFAR10ConvClassifier(dropout_p=0.9)
optimizer = optim.Adam(high_dropout_model.parameters(), lr=LEARNING_RATE)
high_epoch, high_history = train_with_early_stopping(
    high_dropout_model, train_loader, valid_loader, criterion, optimizer, EPOCHS, PATIENCE, device)
_, high_accuracy = evaluate(high_dropout_model, test_loader, criterion, device)
print(f'p=0.9 모델 평가 정확도: {high_accuracy * 100:.2f}%')

드롭아웃 p=0.9 모델 학습


  에포크   1 | 훈련 손실 1.9426 | 검증 손실 1.5709 | 검증 정확도 46.60%


  에포크   5 | 훈련 손실 1.5017 | 검증 손실 1.2268 | 검증 정확도 57.60%


  에포크  10 | 훈련 손실 1.3733 | 검증 손실 1.0786 | 검증 정확도 63.00%


  에포크  15 | 훈련 손실 1.3058 | 검증 손실 1.0396 | 검증 정확도 64.69%


...

  에포크  60 | 훈련 손실 1.0916 | 검증 손실 0.8680 | 검증 정확도 70.05%


  에포크  65 | 훈련 손실 1.0861 | 검증 손실 0.8457 | 검증 정확도 70.73%


  에포크  70 | 훈련 손실 1.0703 | 검증 손실 0.8569 | 검증 정확도 70.37%


  에포크  75 | 훈련 손실 1.0652 | 검증 손실 0.8696 | 검증 정확도 69.96%
  조기 종료: 에포크 75 (최적 에포크 65)


p=0.9 모델 평가 정확도: 70.77%


In [9]:
print(f'{"모델":>12} {"최적 에포크":>11} {"훈련 손실":>10} {"검증 손실":>10} {"평가 정확도":>11}')
print('-' * 60)
for name, epoch, history, accuracy in [('p=0.5 (본문)', base_epoch, base_history, base_accuracy),
                                       ('p=0.9', high_epoch, high_history, high_accuracy)]:
    best = history[epoch - 1]
    print(f'{name:>12} {epoch:11d} {best[1]:10.4f} {best[2]:10.4f} {accuracy * 100:10.2f}%')

print()
print('에포크별 훈련 손실과 검증 손실 (앞 10 에포크)')
print(f'{"에포크":>6} | {"p=0.5 훈련":>10} {"p=0.5 검증":>10} | {"p=0.9 훈련":>10} {"p=0.9 검증":>10}')
print('-' * 62)
for i in range(min(10, len(base_history), len(high_history))):
    print(f'{i + 1:6d} | {base_history[i][1]:10.4f} {base_history[i][2]:10.4f} | '
          f'{high_history[i][1]:10.4f} {high_history[i][2]:10.4f}')

          모델      최적 에포크      훈련 손실      검증 손실      평가 정확도
------------------------------------------------------------
  p=0.5 (본문)          26     0.5849     0.7737      73.34%
       p=0.9          65     1.0861     0.8457      70.77%

에포크별 훈련 손실과 검증 손실 (앞 10 에포크)
   에포크 |   p=0.5 훈련   p=0.5 검증 |   p=0.9 훈련   p=0.9 검증
--------------------------------------------------------------
     1 |     1.6011     1.3207 |     1.9426     1.5709
     2 |     1.3178     1.2194 |     1.6995     1.4249
     3 |     1.1936     1.0664 |     1.6119     1.3296
     4 |     1.0983     0.9941 |     1.5487     1.2720
     5 |     1.0310     0.9526 |     1.5017     1.2268
     6 |     0.9848     0.9207 |     1.4594     1.1738
     7 |     0.9402     0.8993 |     1.4357     1.1461
     8 |     0.9017     0.8770 |     1.4249     1.1501
     9 |     0.8736     0.8605 |     1.3887     1.1022
    10 |     0.8435     0.8174 |     1.3733     1.0786


### 풀이 해설

**첫 번째 물음: p=0.9의 영향**

실행 결과를 보면 예상한 방향과 대체로 맞는다. 특히 눈여겨볼 것은 **훈련 손실과 검증 손실의 관계**다.

`p=0.9` 모델은 훈련 손실이 검증 손실보다 **높다.** 보통은 반대인데, 그 이유는 드롭아웃이 학습 모드에서만 작동하기 때문이다.
훈련할 때는 뉴런의 10%만 쓰는 반쪽짜리 모델로 손실을 재고, 검증할 때는 뉴런을 모두 쓴 온전한 모델로 잰다.
그래서 **드롭아웃이 강할수록 훈련 손실이 불리하게 측정된다.**
이 현상은 5-2절에서 본 "훈련 손실이 검증 손실보다 낮으면 과적합"이라는 일반적인 해석을
드롭아웃이 있을 때는 그대로 적용하면 안 된다는 것을 보여 준다.

또 하나 눈에 띄는 것은 **최적 에포크가 26에서 65로 크게 늦춰졌다**는 점이다.
드롭아웃이 강해 과적합이 늦게 찾아오므로 더 오래 학습해도 검증 손실이 계속 내려간다.
즉 `p`를 키우면 **학습이 느려지는 대가로 과적합 시점이 미뤄진다.**

성능 면에서는 `p=0.9`가 본문 모델보다 떨어진다(70.77% 대 73.34%).
과적합을 막으려다 **모델이 쓸 수 있는 표현력까지 깎아 낸** 결과다.
규제는 과하면 과소적합을 만든다. 드롭아웃의 `p` 역시 조정해야 할 하이퍼파라미터이지 크면 클수록 좋은 값이 아니다.

**두 번째 물음: 왜 완전 연결 계층에 적용하는가**

'특징의 복잡도' 관점에서 두 계층이 다루는 정보가 어떻게 다른지 보면 된다.

**합성곱 계층이 다루는 것은 단순하고 국소적인 특징**이다. 경계, 선, 모서리, 색의 얼룩 같은 것들이다.
이런 특징은 **이웃한 위치끼리 정보가 거의 겹친다.** 어떤 픽셀에 세로 경계가 있으면 바로 옆 픽셀에도 보통 있다.
그래서 특징 지도의 값 하나를 0으로 만들어도 **옆자리가 같은 정보를 가지고 있어 규제 효과가 거의 없다.**
본문 p10이 말한 '특징 공유' 성질 때문에, 합성곱 계층의 뉴런은 애초에 파라미터를 공유하고 있어
특정 뉴런에 과하게 의존할 위험도 낮다.

**완전 연결 계층이 다루는 것은 복잡하고 전역적인 특징**이다. 특징 추출기가 찾아낸 것들을 조합해
'뾰족한 귀 + 털 무늬 + 네 다리 → 고양이' 같은 고수준 판단을 만든다.
여기서는 **뉴런 하나하나가 서로 다른 조합을 담당**하므로 정보가 겹치지 않는다.
그리고 이런 조합은 훈련 데이터에만 있는 우연한 상관관계를 외워 버리기 쉽다.
예를 들어 '초록 배경 + 네 다리'라는 조합에만 의존해 사슴을 맞히는 뉴런이 생길 수 있다.
드롭아웃으로 그 뉴런을 무작위로 꺼 버리면, 모델은 **특정 조합에 기대지 못하고 여러 경로로 판단하도록** 강제된다.
이것이 드롭아웃의 규제 효과다.

**정리하면**, 드롭아웃은 '서로 겹치지 않는 복잡한 특징의 조합'에서 효과가 크고,
'서로 겹치는 단순한 국소 특징'에서는 효과가 적다. 그래서 완전 연결 계층에 주로 적용한다.
덧붙이면, 파라미터의 대부분도 완전 연결 계층에 몰려 있어 과적합의 주된 원인이 거기에 있다.

### 문제 검토

- **★★ [오류] 코드 번호가 잘못되었다.** 지문이 "[코드 5-17]의 완전 연결 계층에는 드롭아웃이 적용되어 있다"고 하는데,
  **[코드 5-17]은 '실행 환경에 따라 장치 객체 생성'이고, 드롭아웃이 적용된 모델 클래스는 [코드 5-16]**이다.
  → **[코드 5-16]으로 고쳐야 한다.**
- **적절성: 적합. 두 물음의 성격이 잘 나뉘어 있다.** 첫 번째는 실험으로 확인하는 물음, 두 번째는 개념을 설명하는 물음이다.
  특히 두 번째 물음에서 **'특징의 복잡도 관점에서'라는 조건을 명시한 것이 좋다.** 조건이 없으면 "과적합을 막으려고" 같은
  피상적인 답에서 멈추는데, 관점을 지정하니 합성곱 계층과 완전 연결 계층이 다루는 정보의 성격을 비교하게 된다.
- **[검토] 첫 번째 물음이 '추측'에서 끝난다.** 추측만 하고 넘어가면 확인할 길이 없는데,
  이 경우 실제로 돌려 보면 예상 밖의 관찰(훈련 손실이 검증 손실보다 높아진다)을 얻을 수 있어 아깝다.
  "추측한 후 실제로 값을 바꿔 학습해 확인해 보자"를 덧붙이면 좋겠다.

**윤문안**

> **5-10**. **[코드 5-16]**의 완전 연결 계층에는 드롭아웃이 적용되어 있다.
> - 만약 드롭아웃의 `p` 인자의 값을 0.5에서 0.9로 극단적으로 높인다면, 학습 과정과 모델의 최종 성능에 어떤 영향을 미칠지
>   추측해 본 후, 실제로 값을 바꿔 학습해 확인해 보자. 이때 훈련 손실과 검증 손실의 관계도 눈여겨보자.
> - 이 드롭아웃은 분류기의 은닉층과 출력층 사이에 적용되어 있는데, 이는 일반적으로 드롭아웃이 적용되는 위치이다.
>   드롭아웃을 합성곱 계층이 아닌 완전 연결 계층의 은닉층과 출력층 사이에 주로 적용하는 이유를 '특징의 복잡도' 관점에서 설명해 보자.

## 연습 문제 5-11 [도전 문제]

> 이번 장에서 CIFAR-10 데이터셋 분류 모델의 성능을 73% 가까이 올려 보았다. 여러 방법을 동원해 분류 성능을 어디까지
> 올릴 수 있는지 도전해 보자. 다음과 같은 방법을 적용해 봐도 좋지만, 이외의 방법도 환영한다.
> - 또 다른 데이터 증강 기법 적용
> - 합성곱 계층의 수 또는 필터 수 조절
> - 특징 지도의 크기 조절
> - 분류기를 구성하는 다층 퍼셉트론 구조 변경

In [10]:
# 개선 1: 데이터 증강을 강화한다 (좌우 반전 + 무작위 잘라내기)
strong_train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # 가장자리를 채운 후 무작위 위치에서 잘라냄
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
strong_set = datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=strong_train_transform)
strong_train_set, _ = random_split(strong_set, [40000, 10000],
                                   generator=torch.Generator().manual_seed(SEED))
strong_train_loader = DataLoader(strong_train_set, batch_size=BATCH_SIZE, shuffle=True)
print(f'강화 증강 훈련 샘플 수: {len(strong_train_set)}')

강화 증강 훈련 샘플 수: 40000


In [11]:
# 개선 2~4: 합성곱 블록을 3개로 늘리고, 필터 수를 늘리고, 배치 정규화를 추가한다
class ImprovedConvBlock(nn.Module):
    """합성곱 두 번 후 풀링하는 블록. 배치 정규화로 학습을 안정시킨다."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)

class ImprovedCIFAR10Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ImprovedConvBlock(3, 64),       # (B, 3, 32, 32) -> (B,  64, 16, 16)
            ImprovedConvBlock(64, 128),     #                -> (B, 128,  8,  8)
            ImprovedConvBlock(128, 256),    #                -> (B, 256,  4,  4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                   # (B, 256, 4, 4) -> (B, 4096)
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

improved_model = ImprovedCIFAR10Classifier()
print(f'본문 모델 파라미터 수:   {sum(p.numel() for p in base_model.parameters()):>10,d}')
print(f'개선 모델 파라미터 수: {sum(p.numel() for p in improved_model.parameters()):>12,d}')

본문 모델 파라미터 수:      532,970
개선 모델 파라미터 수:    3,249,994


In [12]:
print('개선 모델 학습')
torch.manual_seed(SEED)
improved_model = ImprovedCIFAR10Classifier()
optimizer = optim.Adam(improved_model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)
improved_epoch, improved_history = train_with_early_stopping(
    improved_model, strong_train_loader, valid_loader, criterion, optimizer,
    epochs=60, patience=15, device=device, scheduler=scheduler)
_, improved_accuracy = evaluate(improved_model, test_loader, criterion, device)
print(f'개선 모델 평가 정확도: {improved_accuracy * 100:.2f}%')

개선 모델 학습


  에포크   1 | 훈련 손실 1.7886 | 검증 손실 1.3837 | 검증 정확도 49.53%


  에포크   5 | 훈련 손실 0.8849 | 검증 손실 0.8101 | 검증 정확도 71.03%


  에포크  10 | 훈련 손실 0.6028 | 검증 손실 0.5992 | 검증 정확도 79.69%


  에포크  15 | 훈련 손실 0.4440 | 검증 손실 0.4386 | 검증 정확도 85.28%


...

  에포크  30 | 훈련 손실 0.1899 | 검증 손실 0.3750 | 검증 정확도 89.23%


  에포크  35 | 훈련 손실 0.1380 | 검증 손실 0.3772 | 검증 정확도 89.89%


  에포크  40 | 훈련 손실 0.0941 | 검증 손실 0.3840 | 검증 정확도 90.06%


  조기 종료: 에포크 43 (최적 에포크 28)


개선 모델 평가 정확도: 88.72%


In [13]:
print(f'{"모델":>14} {"파라미터 수":>13} {"평가 정확도":>11}')
print('-' * 44)
print(f'{"본문 모델":>14} {sum(p.numel() for p in base_model.parameters()):13,d} {base_accuracy * 100:10.2f}%')
print(f'{"개선 모델":>14} {sum(p.numel() for p in improved_model.parameters()):13,d} {improved_accuracy * 100:10.2f}%')
print()
print('클래스별 정확도 비교')

@torch.no_grad()
def class_accuracy(model):
    model.eval()
    correct = torch.zeros(10)
    total = torch.zeros(10)
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predicted = model(inputs).argmax(dim=1)
        for c in range(10):
            mask = labels == c
            correct[c] += (predicted[mask] == c).sum().item()
            total[c] += mask.sum().item()
    return correct / total * 100

base_class_accuracy = class_accuracy(base_model)
improved_class_accuracy = class_accuracy(improved_model)
print(f'{"클래스":>12} {"본문 모델":>10} {"개선 모델":>10} {"차이":>9}')
print('-' * 46)
for c in range(10):
    diff = improved_class_accuracy[c] - base_class_accuracy[c]
    print(f'{class_names[c]:>12} {base_class_accuracy[c]:9.1f}% {improved_class_accuracy[c]:9.1f}% {diff:+8.1f}%p')

            모델        파라미터 수      평가 정확도
--------------------------------------------
         본문 모델       532,970      73.34%
         개선 모델     3,249,994      88.72%

클래스별 정확도 비교


         클래스      본문 모델      개선 모델        차이
----------------------------------------------
    airplane      78.7%      85.7%     +7.0%p
  automobile      85.0%      94.0%     +9.0%p
        bird      65.5%      81.3%    +15.8%p
         cat      53.6%      81.4%    +27.8%p
        deer      68.0%      89.8%    +21.8%p
         dog      61.8%      83.9%    +22.1%p
        frog      81.1%      94.0%    +12.9%p
       horse      76.7%      89.3%    +12.6%p
        ship      82.6%      95.7%    +13.1%p
       truck      80.4%      92.1%    +11.7%p


### 풀이 해설

네 가지를 함께 적용했다.

1. **데이터 증강 강화** — 좌우 반전에 `RandomCrop(32, padding=4)`를 더했다.
   가장자리를 4픽셀 채운 뒤 무작위 위치에서 32×32를 잘라 내므로, 대상이 조금씩 이동한 이미지가 만들어진다.
   CIFAR-10에서 가장 효과가 큰 증강 기법으로 알려져 있다.
2. **합성곱 계층 수와 필터 수 증가** — 블록을 3개로 늘리고, 각 블록 안에 합성곱을 2개씩 두어 합성곱 계층이 6개가 되었다.
   필터 수도 16, 32에서 64, 128, 256으로 크게 늘렸다. CIFAR-10은 MNIST와 달리 실제 사물 사진이라
   **추출할 고수준 특징이 훨씬 많기 때문**이다. 5-6에서 MNIST에 계층을 늘려도 소용없던 것과 정반대다.
3. **특징 지도 크기 조절** — 마지막 특징 지도를 8×8에서 4×4로 줄여 분류기의 입력 부담을 낮췄다.
4. **배치 정규화 추가와 학습률 스케줄링** — 본문에서 다루지 않은 기법이지만,
   계층이 깊어질수록 학습이 불안정해지므로 함께 쓰는 것이 일반적이다.
   `CosineAnnealingLR`은 학습이 진행될수록 학습률을 서서히 낮춰 마무리를 안정시킨다.

**결과** 정확도가 크게 올랐다. 클래스별로 보면 개선 폭이 고른 것이 아니라,
**본문 모델이 특히 못하던 클래스(고양이, 새, 개 등 동물 클래스)에서 개선 폭이 크다.**
이 클래스들은 형태가 다양하고 서로 닮아 고수준 특징이 필요한데, 얕은 모델은 그것을 뽑아내지 못했던 것이다.

**어디까지 올라가는가** paperswithcode.com에서 확인할 수 있듯이 CIFAR-10의 최고 성능은 99%를 넘는다.
다만 그런 모델은 훨씬 깊은 구조(ResNet 계열)와 훨씬 긴 학습 시간을 요구한다.
여기서 중요한 것은 최고 기록이 아니라 **무엇을 바꾸면 무엇이 좋아지는지 감을 잡는 것**이다.

### 문제 검토

- **적절성: 도전 문제로 적합.** 정답이 없는 열린 문제이고, 네 가지 방향을 제시해 막막하지 않게 했다.
  "이외의 방법도 환영한다"라는 표현도 좋다. 실무에서 성능을 올리는 일이 실제로 이런 모습이다.
- **[검토] 목표 수치가 없다.** "어디까지 올릴 수 있는지"만으로는 어디서 멈춰야 할지 알 수 없다.
  "80%를 먼저 목표로 삼아 보자" 정도의 기준이 있으면 시도의 끝이 생긴다. 실제로 위 구성으로 80%대 중반에 이른다.
- **[검토] 무엇이 효과가 있었는지 확인하게 하면 좋겠다.** 네 가지를 한꺼번에 적용하면 무엇이 얼마나 기여했는지 알 수 없다.
  하나씩 더해 가며 확인하라고 하면 훨씬 배우는 것이 많다. 도전 문제이니 강제할 필요는 없지만 권할 만하다.
- **[검토] 정확도를 비교하는 기준.** 본문에서 73%가 나왔다고 했는데, 이는 시드와 실행 환경에 따라 달라진다.
  독자의 기준 모델 정확도가 다를 수 있으므로, 절대 수치보다 **자신의 기준 모델 대비 향상 폭**으로 보게 하는 편이 안전하다.

**윤문안**

> **5-11**. [도전 문제] 이번 장에서 CIFAR-10 데이터셋 분류 모델의 성능을 73% 가까이 올려 보았다. 여러 방법을 동원해
> 분류 성능을 어디까지 올릴 수 있는지 도전해 보자. 우선 80%를 목표로 삼고, 다음과 같은 방법을 하나씩 적용하면서
> 어떤 방법이 얼마나 도움이 되는지 확인해 보자. 이외의 방법도 환영한다.
> (네 항목은 그대로)

## 연습 문제 5-12 [도전 문제]

> 어쩌다 보니 인류의 절반이 좌우 반전된 형태의 숫자를 사용하게 되었다. MNIST 데이터셋으로 모든 인류가 사용하는
> 숫자를 동시에 분류할 수 있는 숫자 분류기 모델을 만들어 보자.

### 접근

먼저 **왜 그냥은 안 되는지** 짚어야 한다. 5-3절에서 CIFAR-10에 좌우 반전 증강을 적용했으니,
MNIST에도 `RandomHorizontalFlip`을 적용하면 될 것 같다. 하지만 그렇게 하면 문제가 생긴다.

**2와 반전된 5, 또는 5와 반전된 2가 서로 닮기 때문이다.** 좌우 반전 증강을 적용하면 같은 이미지에
서로 다른 정답이 붙을 수 있어 모델이 학습할 수 없게 된다. 본문 p31이 "데이터 증강이 항상 도움이 되는 것은 아니며,
데이터의 성격에 맞아야 한다"고 한 경우가 바로 이것이다.

그래서 **무작위 반전이 아니라, 원본과 반전본을 모두 포함하는 데이터셋**을 만드는 방식으로 접근한다.
세 가지 방법을 비교해 본다.

| 방법 | 설명 |
|---|---|
| A. 무작위 반전 증강 | `RandomHorizontalFlip(p=0.5)`. 예상대로 실패할 것이다 |
| B. 원본 + 반전본 결합 (10 클래스) | 데이터를 두 배로 만들고 정답은 원래 숫자 그대로 |
| C. 원본 + 반전본 결합 (20 클래스) | 반전된 숫자에 별도의 클래스를 부여 |

C는 '반전 여부까지 구분'하므로 문제가 요구한 것보다 많은 일을 하지만,
**모호한 쌍에 서로 다른 정답을 주지 않으므로 학습이 가능하다.** 예측할 때 20개 클래스를 10개로 접으면 된다.

In [14]:
mnist_transform = transforms.ToTensor()
mnist_train = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=mnist_transform)
mnist_test = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=mnist_transform)

class FlippedMNIST(torch.utils.data.Dataset):
    """MNIST 의 원본과 좌우 반전본을 함께 제공하는 데이터셋.

    label_offset 가 0이면 반전본에도 원래 숫자 정답을 주고(10 클래스),
    10이면 반전본에 10을 더한 별도 클래스를 준다(20 클래스).
    """

    def __init__(self, base_dataset, label_offset=0):
        self.base_dataset = base_dataset
        self.label_offset = label_offset

    def __len__(self):
        return len(self.base_dataset) * 2

    def __getitem__(self, index):
        image, label = self.base_dataset[index // 2]
        if index % 2 == 1:                          # 홀수 번째는 좌우 반전본
            image = torch.flip(image, dims=[2])     # 마지막 차원(가로)을 뒤집음
            label = label + self.label_offset
        return image, label

print(f'원본 훈련 샘플 수: {len(mnist_train)}')
print(f'결합 훈련 샘플 수: {len(FlippedMNIST(mnist_train))}')

원본 훈련 샘플 수: 60000
결합 훈련 샘플 수: 120000


In [15]:
class MNISTClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

MNIST_EPOCHS = 5

def train_mnist(model, loader, epochs=MNIST_EPOCHS):
    model.to(device).train()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
    return model

@torch.no_grad()
def accuracy_on(model, loader, fold_20=False):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        predicted = model(images).argmax(dim=1)
        if fold_20:
            predicted = predicted % 10      # 20 클래스 예측을 10 클래스로 접음
            labels = labels % 10
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    return correct / total * 100

In [16]:
# 평가용 데이터로더: 원본만 / 반전본만 / 둘 다
original_test_loader = DataLoader(mnist_test, batch_size=1000)
combined_test_loader_10 = DataLoader(FlippedMNIST(mnist_test, label_offset=0), batch_size=1000)
combined_test_loader_20 = DataLoader(FlippedMNIST(mnist_test, label_offset=10), batch_size=1000)

# 방법 A: 무작위 좌우 반전 증강
flip_transform = transforms.Compose([transforms.RandomHorizontalFlip(p=0.5), transforms.ToTensor()])
flip_train = datasets.MNIST(root=DATA_ROOT, train=True, download=False, transform=flip_transform)
torch.manual_seed(SEED)
print('방법 A: 무작위 좌우 반전 증강 학습')
model_a = train_mnist(MNISTClassifier(10), DataLoader(flip_train, batch_size=64, shuffle=True))
acc_a_original = accuracy_on(model_a, original_test_loader)
acc_a_combined = accuracy_on(model_a, combined_test_loader_10)
print(f'  원본만: {acc_a_original:.2f}%, 원본+반전본: {acc_a_combined:.2f}%')

방법 A: 무작위 좌우 반전 증강 학습


  원본만: 98.23%, 원본+반전본: 98.28%


In [17]:
# 방법 B: 원본 + 반전본을 모두 포함하되 정답은 원래 숫자 (10 클래스)
torch.manual_seed(SEED)
print('방법 B: 원본+반전본 결합, 10 클래스')
model_b = train_mnist(MNISTClassifier(10),
                      DataLoader(FlippedMNIST(mnist_train, 0), batch_size=64, shuffle=True))
acc_b_original = accuracy_on(model_b, original_test_loader)
acc_b_combined = accuracy_on(model_b, combined_test_loader_10)
print(f'  원본만: {acc_b_original:.2f}%, 원본+반전본: {acc_b_combined:.2f}%')

# 방법 C: 반전본에 별도 클래스를 부여 (20 클래스) 후 예측을 10 클래스로 접음
torch.manual_seed(SEED)
print('방법 C: 원본+반전본 결합, 20 클래스')
model_c = train_mnist(MNISTClassifier(20),
                      DataLoader(FlippedMNIST(mnist_train, 10), batch_size=64, shuffle=True))
acc_c_combined_20 = accuracy_on(model_c, combined_test_loader_20)
acc_c_combined_10 = accuracy_on(model_c, combined_test_loader_20, fold_20=True)
print(f'  20 클래스(반전 여부까지): {acc_c_combined_20:.2f}%, 10 클래스로 접었을 때: {acc_c_combined_10:.2f}%')

방법 B: 원본+반전본 결합, 10 클래스


  원본만: 98.58%, 원본+반전본: 98.58%
방법 C: 원본+반전본 결합, 20 클래스


  20 클래스(반전 여부까지): 98.25%, 10 클래스로 접었을 때: 98.81%


In [18]:
print(f'{"방법":>26} {"원본+반전본 정확도":>18}')
print('-' * 48)
print(f'{"A. 무작위 반전 증강":>26} {acc_a_combined:17.2f}%')
print(f'{"B. 결합 (10 클래스)":>26} {acc_b_combined:17.2f}%')
print(f'{"C. 결합 (20 -> 10 클래스)":>26} {acc_c_combined_10:17.2f}%')

# 어떤 숫자에서 틀리는지 확인한다
@torch.no_grad()
def confusion_top(model, loader, top_k=6):
    model.eval()
    errors = {}
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        predicted = model(images).argmax(dim=1)
        for true_label, pred_label in zip(labels[predicted != labels].tolist(),
                                          predicted[predicted != labels].tolist()):
            key = (true_label, pred_label)
            errors[key] = errors.get(key, 0) + 1
    return sorted(errors.items(), key=lambda item: -item[1])[:top_k]

print()
print('방법 B 모델이 가장 많이 틀린 조합 (정답 -> 예측)')
for (true_label, pred_label), count in confusion_top(model_b, combined_test_loader_10):
    print(f'  {true_label} -> {pred_label}: {count}개')

                        방법         원본+반전본 정확도
------------------------------------------------
              A. 무작위 반전 증강             98.28%
            B. 결합 (10 클래스)             98.58%
      C. 결합 (20 -> 10 클래스)             98.81%

방법 B 모델이 가장 많이 틀린 조합 (정답 -> 예측)


  4 -> 9: 28개
  2 -> 5: 16개
  7 -> 2: 14개
  2 -> 6: 12개
  7 -> 9: 12개
  5 -> 2: 10개


### 풀이 해설

**결과부터 보자. 세 방법 모두 잘 된다.**

| 방법 | 원본+반전본 정확도 |
|---|---|
| A. 무작위 반전 증강 | 98.28% |
| B. 결합 (10 클래스) | 98.58% |
| C. 결합 (20 → 10 클래스) | **98.81%** |

셋 다 98%를 넘고 차이는 0.5%p 안쪽이다. **좌우 반전 숫자를 함께 분류하는 일이 생각보다 훨씬 쉽다**는 뜻이다.
이것이 이 문제에서 가장 뜻밖의 관찰이다.

**예상은 왜 빗나갔는가**

흔히 이렇게 예상한다. "2를 좌우로 뒤집으면 5와 닮으니, 둘을 같이 배우면 서로 헷갈려 정확도가 크게 떨어질 것이다."
실제로 틀린 조합을 보면 `2 → 5`(16개)와 `5 → 2`(10개)가 상위에 있어 이 짐작이 아주 틀리지는 않았다.

하지만 가장 많이 틀린 조합은 **`4 → 9`(28개)**다. 이는 좌우 반전과 아무 상관이 없는,
**원래 MNIST에서도 가장 흔한 혼동**이다. 즉 반전 때문에 새로 생긴 오류보다 원래 있던 오류가 더 많다.

왜 2와 뒤집힌 5가 덜 헷갈릴까? **손으로 쓴 2와 5는 획의 구성이 꽤 다르기 때문**이다.
인쇄체로는 비슷해 보여도, 2는 위쪽이 둥근 곡선이고 아래쪽이 곧은 가로획인 반면
5는 위쪽이 곧은 가로획이고 아래쪽이 둥근 곡선이다. 뒤집어도 이 차이는 남는다.
**모델은 우리가 인쇄체를 떠올리며 짐작한 것보다 세밀한 단서를 쓴다.**

방법 C의 결과가 이를 확증한다. **반전 여부까지 맞히는 20 클래스 문제에서도 98.25%가 나온다.**
즉 모델은 '이것이 5인가 뒤집힌 2인가'까지 거의 정확히 구분한다. 두 형태가 원리적으로 겹친다면 불가능한 성적이다.

**세 방법의 성격 차이**

*방법 A(무작위 반전 증강)*는 가장 간단하고, 결과도 나쁘지 않다(98.28%).
다만 **에포크마다 같은 샘플이 원본이 되기도 하고 반전본이 되기도 한다.**
원본만으로 평가하면 98.23%로 B(98.58%)보다 조금 낮은데, 원본을 볼 기회가 절반으로 줄기 때문이다.
'절반의 확률로 뒤집는다'는 것은 데이터를 늘리는 것이 아니라 **바꿔치기하는 것**이다.

*방법 B(원본+반전본 결합, 10 클래스)*는 데이터를 실제로 두 배로 만든다.
정답이 흔들리지 않고 원본과 반전본을 모두 항상 보므로, **원본만 평가해도(98.58%) 결합해 평가해도(98.58%) 같다.**
문제가 요구한 것을 가장 직접적으로 만족하는 방법이다.

*방법 C(20 클래스)*는 반전 여부까지 맞히게 한 뒤 예측을 10 클래스로 접는다.
결과적으로 가장 높은 98.81%가 나왔는데, **모델에 더 자세한 정답을 준 것이 도움이 된 경우**로 볼 수 있다.
'5'와 '뒤집힌 2'를 같은 답으로 묶으라고 강요하지 않으니 모델이 더 깔끔한 경계를 배운다.
다만 **문제가 요구한 것은 숫자를 맞히는 것이지 표기 방식을 구분하는 것이 아니므로**,
'가장 알맞은 답'은 여전히 B다. C는 덤으로 얻은 관찰이다.

**그래도 남는 교훈**

이 문제를 '쉬웠다'로 끝내면 아깝다. 짚어 둘 것이 둘 있다.

**첫째, 데이터 증강은 데이터의 성격에 맞아야 한다.** 5-3절에서 CIFAR-10에 좌우 반전을 적용한 것은
'뒤집힌 고양이도 고양이'라는 사실이 성립하기 때문이다. 숫자에는 그 사실이 성립하지 않는다.
이 문제에서 A가 그럭저럭 통한 것은 **문제 설정 자체가 '뒤집힌 숫자도 같은 숫자'로 바뀌었기 때문**이지,
MNIST에 좌우 반전 증강을 써도 된다는 뜻이 아니다.
평범한 MNIST 분류기에 A를 적용하면 성능이 떨어진다.

**둘째, 예상은 실험으로 확인해야 한다.** '2와 뒤집힌 5가 겹쳐서 어려울 것'이라는 짐작은 그럴듯하지만,
실제로 재 보면 그 영향이 원래 있던 `4 → 9` 혼동보다 작다. 이 확인 과정이 이 문제의 진짜 내용이다.

### 문제 검토

- **적절성: 도전 문제로 적합하다. 그리고 설정이 재미있다.** '인류의 절반이 좌우 반전된 숫자를 쓴다'는
  상황 설정이 딱딱한 문제집 분위기를 덜어 준다. 5장 마지막 문제로 어울린다.
- **★ 이 문제의 도전 지점은 정확도가 아니라 그 앞에 있다.**
  5-3절에서 `RandomHorizontalFlip`은 **데이터 증강**, 즉 실제 분포는 그대로 둔 채 훈련 데이터를 흔들어
  일반화를 돕는 도구로 배웠다. 그런데 이 문제에서는 **실제 분포 자체가 바뀌었다.**
  좌우 반전은 더 이상 '흔들기'가 아니라 **바뀐 분포를 데이터셋에 반영하는 일**이다.
  **같은 함수를 전혀 다른 목적으로 다시 쓸 수 있는가**가 이 문제의 전부다.

  | | 데이터 증강(5-3절) | 이 문제 |
  |---|---|---|
  | 실제 데이터 분포 | 그대로다 | **바뀌었다** |
  | 좌우 반전의 역할 | 훈련 데이터를 흔든다 | **바뀐 분포를 담는다** |
  | 훈련 샘플 수 | 그대로 60,000 | **120,000** |

- **그래서 뒤가 쉽게 풀리는 것은 흠이 아니다.** 방법 B는 원본만 평가해도(98.58%),
  원본과 반전본을 함께 평가해도(98.58%) **같은 정확도**가 나온다.
  데이터셋을 올바로 구성했으므로 두 표기가 모델에게 똑같이 익숙한 것이다.
  **설계가 맞으면 학습은 어려울 것이 없다** — 좋은 설계 문제의 전형적인 모습이다.
- **★ [검토] 다만 위험이 하나 있다. 방법 A로도 거의 맞는다.**
  `RandomHorizontalFlip(p=0.5)`을 훈련 데이터에 그대로 얹는 것이 가장 먼저 떠오르는데,
  이는 **데이터를 늘리는 것이 아니라 바꿔치기하는 것**이다. 그런데도 98.28%가 나와 B와 0.3%p밖에 차이 나지 않는다.
  A로 푼 독자는 **정답을 얻었다고 여기고 이 문제의 핵심을 통과해 버린다.**

  정확도로는 둘이 갈리지 않지만 **훈련 샘플 수는 60,000과 120,000으로 두 배 차이가 난다.**
  → "학습에 사용한 샘플 수가 원래 MNIST 훈련 데이터셋과 어떻게 달라졌는지도 확인해 보자"를 덧붙이면,
  A를 쓴 독자가 `len(dataset)`이 여전히 60,000인 것을 보고 **'절반의 인류는 어디 갔지?' 하고 스스로 되묻게 된다.**
  개념을 미리 말해 주지 않으면서 설계 판단을 되짚게 하는 장치다.
- **[검토] 본문과의 연결이 좋다.** 5-3절이 좌우 반전 증강이 도움이 된 사례(CIFAR-10)를 보여 줬다면,
  이 문제는 같은 기법을 **다른 목적으로 쓰는** 사례다. 도구는 같고 쓰임이 다르다는 것을 보여 주는 배치다.

**윤문안(선택)**

> **5-12**. [도전 문제] 어쩌다 보니 인류의 절반이 좌우 반전된 형태의 숫자를 사용하게 되었다.
> MNIST 데이터셋으로 모든 인류가 사용하는 숫자를 동시에 분류할 수 있는 숫자 분류기 모델을 만들어 보자.
> **학습에 사용한 샘플 수가 원래 MNIST 훈련 데이터셋과 어떻게 달라졌는지도 확인해 보자.**

**넣지 않아도 문제는 성립한다.** 생각하며 푸는 독자는 어차피 A와 B의 차이에 이르고,
지문을 짧게 두는 편이 도전 문제답다는 판단도 타당하다.

**참고**: 흔히 예상하는 '2와 뒤집힌 5의 혼동'은 실제로는 영향이 작다(위 풀이 해설 참고).
이는 관찰거리일 뿐 문제의 흠은 아니다.